# Lab 7 — ETL pipeline with the Medallion architecture and Delta Lake
### From raw files to reliable tables

So far we have worked with clean data. In practice, data arrives **dirty, duplicated, late and in different formats**. Today we build a complete pipeline that turns raw files into tables ready for analysis.

**Objectives**
- Understand data lake, lakehouse and the Medallion architecture (Bronze, Silver, Gold)
- Ingest CSV and JSON with real-world problems
- Apply quality rules and set rejected records aside (quarantine)
- Write partitioned Parquet and take advantage of *partition pruning*
- Use Delta Lake: ACID, MERGE, time travel, schema enforcement, OPTIMIZE and VACUUM

---
## 1. Theory

### Data warehouse, data lake and lakehouse

| | Data Warehouse | Data Lake | Lakehouse |
|-|----------------|-----------|-----------|
| Stores | Structured tables | Files of any type | Open files + table layer |
| Format | Proprietary | CSV, JSON, Parquet... | Parquet + transaction log (Delta, Iceberg, Hudi) |
| Cost | High | Low (object storage) | Low |
| Transactions (ACID) | Yes | No | Yes |
| UPDATE / DELETE | Yes | Hard (rewriting files) | Yes |

The lakehouse combines the low cost of a data lake with the guarantees of a warehouse.

### Medallion architecture

```
  SOURCES            BRONZE                SILVER                 GOLD
  CSV, JSON   ──▶   raw data       ──▶    clean data     ──▶    aggregated data
  APIs, CDC         as it arrived         typed, validated       ready for
                    + metadata            no duplicates          BI / ML / API
                    all strings           business rules
                                     └──▶ QUARANTINE (rejected)
```

| Layer | Question it answers | Golden rule |
|--------|----------------------|---------------|
| **Bronze** | "What exactly arrived?" | Never discard anything. Allows reprocessing. |
| **Silver** | "What is the correct version of each record?" | Right types, one row per entity, rules applied. |
| **Gold** | "What does the business need to see?" | Aggregations and metrics, modeled for consumption. |

### Why Parquet

| | CSV / JSON | Parquet |
|-|-----------|---------|
| Orientation | Row | **Column** |
| Schema | Inferred on every read | Stored in the file |
| Compression | Weak | Strong (per column) |
| Reading 2 of 20 columns | Reads everything | Reads only those 2 (*column pruning*) |
| Filtering `price > 100` | Reads everything | Skips blocks using min/max statistics (*predicate pushdown*) |

---
## 2. Setup

> This lab uses Delta Lake. On Colab or locally:
> `pip install pyspark==4.0.1 delta-spark==4.0.0`
> Creating the session for the first time downloads the Delta jars from Maven (internet required).

In [1]:
# !pip install -q pyspark==4.0.1 delta-spark==4.0.0

In [2]:
import os, json, csv, random, shutil
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (
    SparkSession.builder
    .appName("Lab 7 - ETL Medallion + Delta")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()   # downloads the Delta jars the first time
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:30:15 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:30:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/felipe/anaconda3/envs/pyspark/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-98ffca8f-a95a-46bb-aa88-1cfbb8fb5edc;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 101ms :: artifacts dl 3ms
	:: modul

Spark 4.0.1


In [3]:
LAKE       = "/tmp/lake"
RAW        = f"{LAKE}/raw"
BRONZE     = f"{LAKE}/bronze/orders"
SILVER     = f"{LAKE}/silver/orders"
QUARANTINE = f"{LAKE}/quarantine/orders"
GOLD       = f"{LAKE}/gold"

def tree(path):
    """Shows the folder structure with file counts and sizes."""
    for root, dirs, files in sorted(os.walk(path)):
        dirs.sort()
        data = [f for f in files if not f.startswith((".", "_"))]
        depth = root[len(path):].count(os.sep)
        size = sum(os.path.getsize(os.path.join(root, f)) for f in data) / 1024
        info = f"  ({len(data)} files, {size:,.0f} KB)" if data else ""
        print("    " * depth + (os.path.basename(root) or root) + "/" + info)

def scan_info(df):
    """Shows the filters and columns that reach the file read."""
    import io, contextlib, re
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    scan = next(l for l in buf.getvalue().splitlines() if "FileScan" in l)
    for key in ["PartitionFilters", "PushedFilters", "ReadSchema"]:
        m = re.search(key + r": (\[.*?\]|struct<.*?>)", scan)
        print(f"{key:<17}: {m.group(1) if m else '-'}")

---
## 3. Raw data

We simulate two source systems of a store:

| Source | Format | Quirks |
|--------|---------|------------------|
| Physical store (POS) | CSV | Dates in two formats, prices with "R$" |
| E-commerce app | JSON lines | Extra `device` column, ISO dates |

Problems added on purpose: regions with inconsistent casing/spaces or not provided, invalid prices (`N/A`, negative), zero or empty quantities, corrupted dates, empty customers, statuses outside the domain, nonexistent products, **duplicate rows** and **status updates** for the same order arriving on later days.

In [4]:
REGIONS  = ["North", "Northeast", "Midwest", "Southeast", "South"]
STATUSES = ["created", "paid", "shipped", "delivered", "cancelled"]
PRODUCTS = [
    (1, "Notebook", "Computers", 3500), (2, "Monitor", "Computers", 1200),
    (3, "Mouse", "Peripherals", 85),    (4, "Keyboard", "Peripherals", 150),
    (5, "Webcam", "Peripherals", 320),  (6, "USB Hub", "Peripherals", 120),
    (7, "Headset", "Audio", 250),       (8, "Speaker", "Audio", 400),
    (9, "External HD", "Storage", 400), (10, "Pendrive", "Storage", 60),
    (11, "USB Cable", "Cables", 25),    (12, "HDMI Cable", "Cables", 40),
]

rng = random.Random(42)
next_id = [100_000]
known_orders = {}   # order_id -> created_at (to generate updates on later days)

def dirty_region(r):
    if rng.random() < 0.01: return "N/A"           # region not provided
    return rng.choice([r, r, r, r.lower(), " " + r.upper(), r + "  "])

def dirty_price(p):
    roll = rng.random()
    if roll < 0.03: return "N/A"
    if roll < 0.05: return f"-{p:.2f}"
    if roll < 0.25: return f"R$ {p:.2f}"
    return f"{p:.2f}"

def dirty_quantity():
    roll = rng.random()
    if roll < 0.02: return ""
    if roll < 0.04: return "0"
    return str(rng.randint(1, 4))

def make_order(order_id, created, updated, status, fmt):
    pid, _, _, base = rng.choice(PRODUCTS)
    if rng.random() < 0.02:
        pid = 999                                   # nonexistent product
    ts_created = created.strftime(fmt) if rng.random() > 0.01 else "2024-13-45 99:99"   # corrupted date
    return {
        "order_id":    str(order_id),
        "customer_id": str(rng.randint(1, 3000)) if rng.random() > 0.01 else "",
        "product_id":  str(pid),
        "quantity":    dirty_quantity(),
        "price":       dirty_price(base * rng.uniform(0.9, 1.1)),
        "region":      dirty_region(rng.choice(REGIONS)),
        "status":      status if rng.random() > 0.01 else "pending_review",       # status outside the domain
        "created_at":  ts_created,
        "updated_at":  updated.strftime(fmt),
    }

def generate_day(day, n_new=1200, n_updates=300):
    """Generates one day's files: new orders + updates to older orders."""
    rows_csv, rows_json = [], []
    for _ in range(n_new):
        oid = next_id[0]; next_id[0] += 1
        created = day + timedelta(minutes=rng.randint(0, 1439))
        is_app = rng.random() < 0.6
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else rng.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"])
        row = make_order(oid, created, created, rng.choice(["created", "paid"]), fmt)
        known_orders[oid] = (created, is_app, row)
        (rows_json if is_app else rows_csv).append(row)

    # status updates for orders from previous days
    old = [o for o, (c, _, _) in known_orders.items() if c.date() < day.date()]
    for oid in rng.sample(old, min(n_updates, len(old))):
        created, is_app, original = known_orders[oid]
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else "%Y-%m-%d %H:%M:%S"
        row = dict(original)
        row["status"] = rng.choice(["shipped", "delivered", "cancelled"])
        row["created_at"] = created.strftime(fmt)
        row["updated_at"] = (day + timedelta(minutes=rng.randint(0, 1439))).strftime(fmt)
        (rows_json if is_app else rows_csv).append(row)

    # exact duplicates (file resent, API retry...)
    rows_csv  += rng.sample(rows_csv,  len(rows_csv)  // 30)
    rows_json += rng.sample(rows_json, len(rows_json) // 30)

    tag = day.strftime("%Y%m%d")
    with open(f"{RAW}/orders_csv/pos_{tag}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows_csv[0].keys()))
        w.writeheader(); w.writerows(rows_csv)
    with open(f"{RAW}/orders_json/app_{tag}.json", "w") as f:
        for r in rows_json:
            r = dict(r, device=rng.choice(["ios", "android", "web"]))
            if rng.random() < 0.5 and r["price"].replace(".", "").isdigit():
                r["price"] = float(r["price"])      # in JSON, sometimes a number, sometimes text
            f.write(json.dumps(r) + "\n")
    return len(rows_csv), len(rows_json)

shutil.rmtree(LAKE, ignore_errors=True)
os.makedirs(f"{RAW}/orders_csv"); os.makedirs(f"{RAW}/orders_json")

with open(f"{RAW}/products.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["product_id", "product_name", "category", "list_price"]); w.writerows(PRODUCTS)

for d in ["2024-07-01", "2024-07-02", "2024-07-03"]:
    n_csv, n_json = generate_day(datetime.fromisoformat(d))
    print(f"{d}: {n_csv} CSV rows, {n_json} JSON rows")

tree(RAW)

2024-07-01: 505 CSV rows, 734 JSON rows
2024-07-02: 587 CSV rows, 962 JSON rows
2024-07-03: 584 CSV rows, 965 JSON rows
raw/  (1 files, 0 KB)
    orders_csv/  (3 files, 126 KB)
    orders_json/  (3 files, 608 KB)


In [5]:
# A look at the files as they are
print(open(f"{RAW}/orders_csv/pos_20240701.csv").read()[:600])
print(open(f"{RAW}/orders_json/app_20240701.json").readline())

order_id,customer_id,product_id,quantity,price,region,status,created_at,updated_at
100001,,8,4,R$ 387.22,midwest,created,01/07/2024 22:10,01/07/2024 22:10
100002,1201,9,3,406.19,North,paid,2024-07-01 11:44:00,2024-07-01 11:44:00
100003,859,8,1,R$ 408.73,northeast,paid,01/07/2024 07:56,01/07/2024 07:56
100004,1097,1,3,3298.84,Southeast,paid,2024-07-01 21:50:00,2024-07-01 21:50:00
100007,627,11,2,25.20, SOUTH,created,2024-07-01 08:59:00,2024-07-01 08:59:00
100013,241,7,1,274.27,Southeast  ,created,2024-07-01 00:30:00,2024-07-01 00:30:00
100014,2221,4,3,136.71, SOUTH,created,01/07/2024 18:58,01/0
{"order_id": "100000", "customer_id": "2234", "product_id": "4", "quantity": "4", "price": "R$ 135.95", "region": "Northeast", "status": "paid", "created_at": "2024-07-01T21:49:00", "updated_at": "2024-07-01T21:49:00", "device": "ios"}



---
## 4. Bronze: ingest without losing anything

Rules of the Bronze layer:
- Read **everything as strings**. If a price comes in as `N/A`, the read must not fail or silently turn the value into a null.
- Combine the sources while preserving columns that exist in only one of them (`unionByName(..., allowMissingColumns=True)`).
- Add **lineage metadata**: which file it came from and when it was ingested.

> `inferSchema=True` is convenient but dangerous in pipelines: the schema can change from one day to the next depending on the incoming data.

In [6]:
raw_csv = (
    spark.read
    .option("header", True)                 # no inferSchema: everything is a string
    .csv(f"{RAW}/orders_csv")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("store"))
)

raw_json = (
    spark.read
    .option("primitivesAsString", True)     # numbers and booleans also become strings
    .json(f"{RAW}/orders_json")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("app"))
)

bronze = (
    raw_csv.unionByName(raw_json, allowMissingColumns=True)
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze.write.mode("overwrite").parquet(BRONZE)
bronze = spark.read.parquet(BRONZE)

bronze.printSchema()
bronze.groupBy("channel").count().show()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- device: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

+-------+-----+
|channel|count|
+-------+-----+
|    app| 2661|
|  store| 1676|
+-------+-----+



`_metadata` is a hidden column Spark provides on file reads (path, size, modification date). It replaces the old `input_file_name()`.

---
## 5. Silver: clean, validate and deduplicate

The Silver work splits into four steps, each in its own function. Small, reusable functions let us process new days with the same code (Lab 8).

```
  bronze ──▶ clean() ──▶ enrich() ──▶ validate() ──┬──▶ deduplicate() ──▶ SILVER
             types        join with                 │
             formats      products                  └──▶ QUARANTINE (+ reason)
```

### ANSI mode in Spark 4

In Spark 4, **ANSI mode is on by default**: `cast("abc" as int)` **raises an error** instead of returning null. On dirty data, use the `try_` variants:

| Operation | Fails on invalid value | Returns null |
|----------|-------------------------|--------------|
| Type conversion | `col.cast("int")` | `col.try_cast("int")` |
| Text to timestamp | `F.to_timestamp(c, fmt)` | `F.try_to_timestamp(c, F.lit(fmt))` |
| Division | `a / b` | `F.try_divide(a, b)` |

In [7]:
# ANSI mode demo
demo = spark.createDataFrame([("10",), ("N/A",)], ["v"])
demo.select("v", F.col("v").try_cast("int").alias("try_cast")).show()

import logging
quiet = logging.getLogger("DataFrameQueryContextLogger")
spark.sparkContext.setLogLevel("OFF"); quiet.setLevel(logging.CRITICAL)   # just to keep the output clean

try:
    demo.select(F.col("v").cast("int")).show()
except Exception as e:
    print("cast() failed:", str(e).split(".")[0])

spark.sparkContext.setLogLevel("ERROR"); quiet.setLevel(logging.NOTSET)

+---+--------+
|  v|try_cast|
+---+--------+
| 10|      10|
|N/A|    NULL|
+---+--------+

cast() failed: [CAST_INVALID_INPUT] The value 'N/A' of the type "STRING" cannot be cast to "INT" because it is malformed


In [8]:
TS_FORMATS = ["yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm", "yyyy-MM-dd'T'HH:mm:ss"]

def parse_ts(col):
    """Tries each format and keeps the first one that works."""
    return F.coalesce(*[F.try_to_timestamp(F.col(col), F.lit(fmt)) for fmt in TS_FORMATS])

def clean(df):
    return (
        df
        .withColumn("order_id",    F.col("order_id").try_cast("long"))
        .withColumn("customer_id", F.col("customer_id").try_cast("long"))
        .withColumn("product_id",  F.col("product_id").try_cast("int"))
        .withColumn("quantity",    F.col("quantity").try_cast("int"))
        .withColumn("price",       F.regexp_replace("price", r"R\$|\s", "").try_cast("double"))
        .withColumn("region",      F.initcap(F.trim("region")))
        .withColumn("status",      F.lower(F.trim("status")))
        .withColumn("created_at",  parse_ts("created_at"))
        .withColumn("updated_at",  parse_ts("updated_at"))
    )

products = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW}/products.csv")

def enrich(df):
    return (
        df.join(F.broadcast(products.select("product_id", "product_name", "category")), "product_id", "left")
        .withColumn("revenue", F.round(F.col("price") * F.col("quantity"), 2))
        .withColumn("order_date", F.to_date("created_at"))
    )

clean(bronze).select("order_id", "price", "quantity", "region", "created_at").show(5)

+--------+-------+--------+---------+-------------------+
|order_id|  price|quantity|   region|         created_at|
+--------+-------+--------+---------+-------------------+
|  102400| 260.16|       4|Southeast|2024-07-03 23:37:00|
|  102402|3314.45|       1|Northeast|2024-07-03 04:19:00|
|  102404| 251.04|       4|Southeast|2024-07-03 01:19:00|
|  102405| 267.12|       3|Northeast|2024-07-03 11:20:00|
|  102406| 439.52|       1|Southeast|2024-07-03 10:15:00|
+--------+-------+--------+---------+-------------------+
only showing top 5 rows


### Quality rules and quarantine

Instead of deleting bad records with `filter`, we keep the rejected ones **along with the reason**. That lets us measure source quality, notify the source team and reprocess after a fix.

In [9]:
RULES = {
    "null_order_id":        F.col("order_id").isNull(),
    "invalid_price":        F.col("price").isNull() | (F.col("price") <= 0),
    "invalid_quantity":     F.col("quantity").isNull() | (F.col("quantity") <= 0),
    "invalid_date":         F.col("created_at").isNull() | F.col("updated_at").isNull(),
    "unknown_region":       F.col("region").isNull() | ~F.col("region").isin(REGIONS),
    "unknown_product":      F.col("category").isNull(),
}

def validate(df):
    errors = F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in RULES.items()]))
    df = df.withColumn("_errors", errors)
    valid    = df.filter(F.size("_errors") == 0).drop("_errors")
    rejected = df.filter(F.size("_errors") > 0)
    return valid, rejected

def deduplicate(df):
    """Keeps only the latest version of each order."""
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

SILVER_COLS = ["order_id", "customer_id", "product_id", "product_name", "category",
               "quantity", "price", "revenue", "region", "status", "channel", "device",
               "created_at", "updated_at", "order_date", "_source_file", "_ingested_at"]

valid, rejected = validate(enrich(clean(bronze)))
silver_df = deduplicate(valid).select(SILVER_COLS)

print(f"Bronze    : {bronze.count():>6,} rows")
print(f"Valid     : {valid.count():>6,} rows")
print(f"Rejected  : {rejected.count():>6,} rows")
print(f"Silver    : {silver_df.count():>6,} unique orders")

Bronze    :  4,337 rows
Valid     :  3,809 rows
Rejected  :    528 rows
Silver    :  3,156 unique orders


In [10]:
# Why were the records rejected?
(
    rejected
    .select(F.explode("_errors").alias("reason"))
    .groupBy("reason").count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+----------------+-----+
|reason          |count|
+----------------+-----+
|invalid_price   |202  |
|invalid_quantity|160  |
|unknown_product |89   |
|unknown_region  |59   |
|invalid_date    |29   |
+----------------+-----+



### Partitioned writes

`partitionBy("order_date")` creates one folder per date value. Queries that filter by date read only the folders they need (*partition pruning*).

Good partition columns have **low cardinality** and are **used in filters** (date, region). Partitioning by `order_id` would create millions of tiny folders.

In [11]:
(silver_df
 .repartition(8)                     # simulates a job with 8 tasks in the last stage
 .write.mode("overwrite")
 .partitionBy("order_date")
 .parquet(SILVER))

rejected.write.mode("overwrite").parquet(QUARANTINE)

tree(f"{LAKE}/silver")

silver/
    orders/
        order_date=2024-07-01/  (8 files, 84 KB)
        order_date=2024-07-02/  (8 files, 83 KB)
        order_date=2024-07-03/  (8 files, 83 KB)


Look at the number of files per partition. Each task in the last stage writes **one file per date partition it encounters**. With 8 tasks, up to 8 files per day appear. This is the **small files** problem: thousands of files of a few KB make reads slow.

The fix is to organize the data by partition before writing:

In [12]:
(silver_df
 .repartition("order_date")          # all of a day's rows in the same task
 .write.mode("overwrite")
 .partitionBy("order_date")
 .parquet(SILVER))

tree(f"{LAKE}/silver")

silver/
    orders/
        order_date=2024-07-01/  (1 files, 42 KB)
        order_date=2024-07-02/  (1 files, 40 KB)
        order_date=2024-07-03/  (1 files, 40 KB)


### Partition pruning, predicate pushdown and column pruning

All three mechanisms appear in the read plan (`FileScan parquet`):

| Mechanism | Field in the plan | Effect |
|-----------|----------------|--------|
| Partition pruning | `PartitionFilters` | Skips whole folders |
| Predicate pushdown | `PushedFilters` | Skips blocks inside files |
| Column pruning | `ReadSchema` | Reads only the columns used |

In [13]:
silver = spark.read.parquet(SILVER)

q = (silver
     .filter(F.col("order_date") == "2024-07-02")   # partition column
     .filter(F.col("price") > 1000)                 # regular column
     .select("order_id", "price", "region"))

scan_info(q)

PartitionFilters : [isnotnull(order_date#340), (order_date#340 = 2024-07-02)]
PushedFilters    : [IsNotNull(price), GreaterThan(price,1000.0)]
ReadSchema       : struct<order_id:bigint,price:double,region:string>


### Reprocessing a single partition

If July 3 needs to be reprocessed, we do not want to rewrite the whole table. With `partitionOverwriteMode=dynamic`, `overwrite` replaces **only the partitions present in the DataFrame**.

In [14]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

before = spark.read.parquet(SILVER).groupBy("order_date").count().orderBy("order_date")
before.show()

day3 = silver_df.filter(F.col("order_date") == "2024-07-03").limit(100)   # simulates a reprocessing run
day3.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(SILVER)

spark.read.parquet(SILVER).groupBy("order_date").count().orderBy("order_date").show()

# back to the full state
silver_df.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(SILVER)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

+----------+-----+
|order_date|count|
+----------+-----+
|2024-07-01| 1055|
|2024-07-02| 1050|
|2024-07-03| 1051|
+----------+-----+

+----------+-----+
|order_date|count|
+----------+-----+
|2024-07-01| 1055|
|2024-07-02| 1050|
|2024-07-03|  100|
+----------+-----+



> With the default mode (`static`), the same command would have **deleted July 1 and 2**. This is one of the most common accidents in plain Parquet pipelines.

---
## 6. Gold: tables for the business

Gold answers specific questions. Cancelled orders do not count toward revenue.

In [15]:
silver = spark.read.parquet(SILVER)
sales  = silver.filter(F.col("status") != "cancelled")

gold_daily = (
    sales.groupBy("order_date", "region", "category")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("revenue"), 2).alias("revenue"),
        F.round(F.avg("revenue"), 2).alias("avg_ticket"),
    )
)

gold_products = (
    sales.groupBy("product_name", "category")
    .agg(F.sum("quantity").alias("units"), F.round(F.sum("revenue"), 2).alias("revenue"))
    .withColumn("rank", F.dense_rank().over(Window.orderBy(F.desc("revenue"))))
)

gold_daily.write.mode("overwrite").partitionBy("order_date").parquet(f"{GOLD}/daily_sales")
gold_products.write.mode("overwrite").parquet(f"{GOLD}/product_ranking")

spark.read.parquet(f"{GOLD}/product_ranking").orderBy("rank").show(5)
tree(LAKE)

+------------+-----------+-----+----------+----+
|product_name|   category|units|   revenue|rank|
+------------+-----------+-----+----------+----+
|    Notebook|  Computers|  579|2024677.91|   1|
|     Monitor|  Computers|  651| 781628.74|   2|
| External HD|    Storage|  607| 242759.35|   3|
|     Speaker|      Audio|  567| 226709.98|   4|
|      Webcam|Peripherals|  562|  181318.1|   5|
+------------+-----------+-----+----------+----+
only showing top 5 rows
lake/
    bronze/
        orders/  (6 files, 151 KB)
    gold/
        daily_sales/
            order_date=2024-07-01/  (1 files, 2 KB)
            order_date=2024-07-02/  (1 files, 2 KB)
            order_date=2024-07-03/  (1 files, 2 KB)
        product_ranking/  (1 files, 2 KB)
    quarantine/
        orders/  (6 files, 56 KB)
    raw/  (1 files, 0 KB)
        orders_csv/  (3 files, 126 KB)
        orders_json/  (3 files, 608 KB)
    silver/
        orders/
            order_date=2024-07-01/  (1 files, 42 KB)
            order

---
## 7. Exercises

In [16]:
# ── Exercise 1 ──────────────────────────────────────────────────────────────
# Add two quality rules to the RULES dictionary:
#   - "null_customer": null customer_id
#   - "invalid_status": status not in STATUSES
# Run validate again and show how many rows each rule rejects.
# Hint: a row can break more than one rule.

# Your solution:
RULES_V2 = {
    **RULES,
    "null_customer":  F.col("customer_id").isNull(),
    "invalid_status": F.col("status").isNull() | ~F.col("status").isin(STATUSES),
}

def validate_v2(df):
    errors = F.array_compact(F.array(*[F.when(c, F.lit(n)) for n, c in RULES_V2.items()]))
    df = df.withColumn("_errors", errors)
    return df.filter(F.size("_errors") == 0).drop("_errors"), df.filter(F.size("_errors") > 0)

_, rejected_v2 = validate_v2(enrich(clean(bronze)))
(rejected_v2.select(F.explode("_errors").alias("reason"))
 .groupBy("reason").count().orderBy(F.desc("count")).show(truncate=False))

+----------------+-----+
|reason          |count|
+----------------+-----+
|invalid_price   |202  |
|invalid_quantity|160  |
|unknown_product |89   |
|unknown_region  |59   |
|invalid_status  |45   |
|null_customer   |41   |
|invalid_date    |29   |
+----------------+-----+



In [17]:
# ── Exercise 2 ──────────────────────────────────────────────────────────────
# Create a Gold table "status funnel by channel":
#   for each channel and status, the number of orders and the percentage within the channel.
# Save it to f"{GOLD}/status_funnel".

# Your solution:
funnel = (
    spark.read.parquet(SILVER)
    .groupBy("channel", "status").count()
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(Window.partitionBy("channel")) * 100, 1))
    .orderBy("channel", F.desc("count"))
)
funnel.write.mode("overwrite").parquet(f"{GOLD}/status_funnel")
funnel.show()

+-------+--------------+-----+----+
|channel|        status|count| pct|
+-------+--------------+-----+----+
|    app|          paid|  850|43.9|
|    app|       created|  768|39.7|
|    app|     delivered|  109| 5.6|
|    app|     cancelled|   98| 5.1|
|    app|       shipped|   91| 4.7|
|    app|pending_review|   19| 1.0|
|  store|       created|  508|41.6|
|  store|          paid|  499|40.9|
|  store|       shipped|   77| 6.3|
|  store|     delivered|   66| 5.4|
|  store|     cancelled|   59| 4.8|
|  store|pending_review|   12| 1.0|
+-------+--------------+-----+----+



In [18]:
# ── Exercise 3 ──────────────────────────────────────────────────────────────
# Write gold_daily partitioned by "region" and prove, from the execution plan,
# that a query filtering on region = 'South' does partition pruning.

# Your solution:
gold_daily.write.mode("overwrite").partitionBy("region").parquet(f"{GOLD}/daily_by_region")

q = spark.read.parquet(f"{GOLD}/daily_by_region").filter(F.col("region") == "South")
scan_info(q)
tree(f"{GOLD}/daily_by_region")

PartitionFilters : [isnotnull(region#659), (region#659 = South)]
PushedFilters    : []
ReadSchema       : struct<order_date:date,category:string,orders:bigint,revenue:double,avg_ticket:double>
daily_by_region/
    region=Midwest/  (1 files, 2 KB)
    region=North/  (1 files, 2 KB)
    region=Northeast/  (1 files, 2 KB)
    region=South/  (1 files, 2 KB)
    region=Southeast/  (1 files, 2 KB)


In [19]:
spark.stop()